# AES-GCM payload encryption and authentication

These cells use **recorded hardware evidence** by default. They do not program a board.
Use the configured Python (SCU35 PMC) kernel. Fresh hardware commands are shown in Markdown and must be invoked explicitly after configuring your exact SCU35 and recovery image.
Current PL timing uses AXI Timer; native PMC timing uses `mcycle`. JTAG/UART are outside operation timing.

The public diagnostic key is bytes `00..1f` in volatile key storage. Exact captured ciphertext/tag and modified-tag rejection are checked by the strict parser. Compare setup + operation time with Python; do not compare transport time to engine throughput.

In [1]:
from pathlib import Path
import hashlib, json, sys, time
import pandas as pd
# Works from the notebook directory, lab root, or parent repository.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "scu35") if (p / "lab.py").is_file()), None)
assert ROOT is not None, "Launch Jupyter inside this checkout or its scu35 directory"
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "host"))
from compare import parse_capture
from lab import config
capture = parse_capture(ROOT / "evidence/timer/uart.txt", "7d1bf77bc454dd1f")
rows = pd.DataFrame(capture["rows"])
print("Recorded board capture: 31 complete rows; current Python:", sys.version.split()[0])


Recorded board capture: 31 complete rows; current Python: 3.11.9


In [2]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag
key = bytes(range(32)); nonce = bytes(range(12)); message = b"SCU35 public demo payload"
ciphertext_tag = AESGCM(key).encrypt(nonce, message, None)
assert AESGCM(key).decrypt(nonce, ciphertext_tag, None) == message
try:
    AESGCM(key).decrypt(nonce, ciphertext_tag[:-1] + bytes([ciphertext_tag[-1] ^ 1]), None)
    raise AssertionError("Modified tag accepted")
except InvalidTag:
    print("Python modified-tag rejection: PASS")
display(rows[rows.algorithm.str.startswith("AES")])
print("Captured hardware AES cases independently checked:", len(capture["aes"]))

Python modified-tag rejection: PASS


,algorithm,case,size_bytes,repeats,copy_cycles,setup_cycles,api_cycles,check_cycles,scu35_api_mean_us,scu35_full_mean_us,scu35_operation_mean_us,scu35_min_us,scu35_max_us,status
10,AES-GCM-ENC,valid,16,20,9180,34368,81300,0,40.6500,62.4240,57.8340,40.65,40.65,0
11,AES-GCM-DEC,valid,16,20,0,34388,69500,9780,34.7500,56.8340,51.9440,34.75,34.75,0
12,AES-GCM-ENC,valid,64,20,34140,34369,81300,0,40.6500,74.9045,57.8345,40.65,40.65,0
13,AES-GCM-DEC,valid,64,20,0,34387,69500,36660,34.7500,70.2735,51.9435,34.75,34.75,0
14,AES-GCM-ENC,valid,256,20,133980,34371,82730,0,41.3650,125.5405,58.5505,41.36,41.37,0
15,AES-GCM-DEC,valid,256,20,0,34393,70932,144180,35.4660,124.7525,52.6625,35.46,35.47,0
16,AES-GCM-ENC,valid,1024,20,533340,34369,89040,0,44.5200,328.3745,61.7045,44.51,44.53,0
17,AES-GCM-DEC,valid,1024,20,0,34391,77244,574260,38.6220,342.9475,55.8175,38.62,38.63,0
18,AES-GCM-ENC,valid,2048,20,1065820,34371,97463,0,48.7315,598.8270,65.9170,48.71,48.74,0
19,AES-GCM-DEC,valid,2048,20,0,34395,85658,1147700,42.8290,633.8765,60.0265,42.82,42.84,0


Captured hardware AES cases independently checked: 5


## Fresh hardware: explicit step

Configure `config.json` first. Run `python lab.py hardware --pdi /path/to/build/demo.pdi` from the lab directory for the full runtime suite, or the native/boot command above for that path. Save logs, verify recovery and compare results before discussing speed. See [audit checkpoints](../docs/audit.md).